# GENERATE WEEKLY REPORT

##### Install required packages
%pip install python-docx pandas openpyxl numpy

### Section 1: Setup and Imports

In [16]:
# =============================================================================
# SECTION 1: SETUP AND IMPORTS
# =============================================================================
# This section installs and imports all required libraries for:
# - Document creation (python-docx)
# - Data processing (pandas, numpy)
# - File operations (os, datetime)

import pandas as pd
import numpy as np
from docx import Document
from docx.shared import Inches, Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.style import WD_STYLE_TYPE
from docx.oxml.ns import qn, nsdecls
from docx.oxml import parse_xml
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# from scipy import stats
# For Section 5
try:
    from scipy import stats
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'scipy', '--break-system-packages', '-q'])
    from scipy import stats

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


### Section 2: Configuration and Path Setup

In [17]:
# =============================================================================
# SECTION 2: CONFIGURATION AND PATH SETUP
# =============================================================================
# Define all paths and configuration settings

# Base path configuration
BASE_PATH = os.path.dirname(os.getcwd())

# Folder paths
OUTPUTS_PATH = os.path.join(BASE_PATH, 'outputs')
VISUALIZATIONS_PATH = os.path.join(BASE_PATH, 'visualizations')
REPORTS_PATH = os.path.join(BASE_PATH, 'reports')
REF_DATE_FILE = os.path.join(BASE_PATH, 'analysis_ref_date.csv')

os.makedirs(REPORTS_PATH, exist_ok=True)

# Configuration settings
CONFIG = {
    'weeks_for_average': 6,
    'spike_threshold_pct': 25,
    'spike_threshold_std': 2,
    'chart_width_inches': 4.0,
    'charts_per_page': 2,
    
    # Trend analysis periods
    'trend_short_term': 6,
    'trend_medium_term': 12,
    'trend_long_term': 24,
}

# Metrics with INVERTED color logic (lower is better)
# For these metrics: going DOWN = favorable (green), going UP = unfavorable (red)
INVERTED_METRICS = [
    'Active Unsubs', 'Auto Unsubs', 'Unsubscribes', 'Unsubscribed',
    'Leak Rate', 'List Unsub Rate', 'Blended Unsub Rate', 'Unsubscribe Rate',
    'Bounce Rate', 'Deal Unsubscribe Rate', 'Off-Market Unsubscribe Rate',
    'Podcast Unsubscribe Rate', 'Case Study Unsubscribe Rate'
]

# Metrics that should use absolute values (stored as negative in CSV)
ABSOLUTE_VALUE_METRICS = [
    'Unsubscribed', 'Unsubscribes'
]

print(f"✓ Configuration loaded")
print(f"  Chart width: {CONFIG['chart_width_inches']}\"")
print(f"  Inverted metrics: {len(INVERTED_METRICS)}")

✓ Configuration loaded
  Chart width: 4.0"
  Inverted metrics: 13


### Section 3: Chart and Data Source Mapping

In [18]:
# =============================================================================
# SECTION 3: CHART AND DATA SOURCE MAPPING
# =============================================================================
# This mapping connects each chart to:
# - 'image': path to PNG file (relative to visualizations folder)
# - 'csv': path to data CSV file (relative to outputs folder)
# - 'metric': column name in CSV containing the metric values
# - 'date_col': column(s) for date reference

CHART_CONFIG = {
    # =========================================================================
    # NEWSLETTER METRICS - Landing Page Funnel (Pages 4-5)
    # =========================================================================
    'Newsletter Visits': {
        'image': 'newsletter_weekly/weekly_visits.png',
        'csv': 'newsletter/weekly_newsletter.csv',
        'metric': 'Visits',
        'date_col': 'Date Range'
    },
    'Newsletter Visit Duration': {
        'image': 'newsletter_weekly/weekly_avg_duration.png',
        'csv': 'newsletter/weekly_newsletter.csv',
        'metric': 'Avg Duration',
        'date_col': 'Date Range'
    },
    'Newsletter Bounce Rate': {
        'image': 'newsletter_weekly/weekly_bounce_rate.png',
        'csv': 'newsletter/weekly_newsletter.csv',
        'metric': 'Bounce Rate',
        'date_col': 'Date Range'
    },
    'Newsletter CVR %': {
        'image': 'general_newsletter_weekly/weekly_newsletter_cvr.png',
        'csv': 'general_newsletter/weekly_table.csv',
        'metric': 'Newsletter CVR %',
        'date_col': 'Date Range'
    },
    
    # =========================================================================
    # NEWSLETTER METRICS - Engagement (Pages 6-7)
    # =========================================================================
    'Blended Open Rate': {
        'image': 'newsletter_series_weekly/blended_open_rate.png',
        'csv': 'newsletter_series/06_blended_weekly.csv',
        'metric': 'Open Rate',
        'date_col': ['Start Date', 'End Date']
    },
    'Blended Verified Click-Through Rate': {
        'image': 'newsletter_series_weekly/blended_verified_ctr.png',
        'csv': 'newsletter_series/06_blended_weekly.csv',
        'metric': 'Verified Click-Through Rate',
        'date_col': ['Start Date', 'End Date']
    },
    'Blended Unsub Rate': {
        'image': 'newsletter_series_weekly/blended_unsubscribe_rate.png',
        'csv': 'newsletter_series/06_blended_weekly.csv',
        'metric': 'Unsubscribe Rate',
        'date_col': ['Start Date', 'End Date']
    },
    
    # =========================================================================
    # NEWSLETTER METRICS - Growth & Churn (Pages 8-9)
    # =========================================================================
    'Growth Rate': {
        'image': 'general_newsletter_weekly/weekly_growth_rate.png',
        'csv': 'general_newsletter/weekly_table.csv',
        'metric': 'Growth Rate',
        'date_col': 'Date Range'
    },
    'New Subscribers': {
        'image': 'general_newsletter_weekly/weekly_subscribed.png',
        'csv': 'general_newsletter/weekly_table.csv',
        'metric': 'Subscribed',
        'date_col': 'Date Range'
    },
    'Unsubscribes': {
        'image': 'general_newsletter_weekly/weekly_unsubscribed.png',
        'csv': 'general_newsletter/weekly_table.csv',
        'metric': 'Unsubscribed',
        'date_col': 'Date Range'
    },
    'Leak Rate': {
        'image': 'general_newsletter_weekly/weekly_leak_rate.png',
        'csv': 'general_newsletter/weekly_table.csv',
        'metric': 'Leak Rate',
        'date_col': 'Date Range'
    },
    
    # =========================================================================
    # NEWSLETTER SERIES - New Deals (Page 10)
    # =========================================================================
    'Deal Open Rate': {
        'image': 'newsletter_series_weekly/deal_open_rate.png',
        'csv': 'newsletter_series/02_deal_emails.csv',
        'metric': 'Open Rate',
        'date_col': 'Date'
    },
    'Deal Verified CTR': {
        'image': 'newsletter_series_weekly/deal_verified_ctr.png',
        'csv': 'newsletter_series/02_deal_emails.csv',
        'metric': 'Verified Click-Through Rate',
        'date_col': 'Date'
    },
    'Deal Unsubscribe Rate': {
        'image': 'newsletter_series_weekly/deal_unsubscribe_rate.png',
        'csv': 'newsletter_series/02_deal_emails.csv',
        'metric': 'Unsubscribe Rate',
        'date_col': 'Date'
    },
    
    # =========================================================================
    # NEWSLETTER SERIES - Off-Market (Page 11)
    # =========================================================================
    'Off-Market Open Rate': {
        'image': 'newsletter_series_weekly/offmarket_open_rate.png',
        'csv': 'newsletter_series/03_offmarket_emails.csv',
        'metric': 'Open Rate',
        'date_col': 'Date'
    },
    'Off-Market Verified CTR': {
        'image': 'newsletter_series_weekly/offmarket_verified_ctr.png',
        'csv': 'newsletter_series/03_offmarket_emails.csv',
        'metric': 'Verified Click-Through Rate',
        'date_col': 'Date'
    },
    'Off-Market Unsubscribe Rate': {
        'image': 'newsletter_series_weekly/offmarket_unsubscribe_rate.png',
        'csv': 'newsletter_series/03_offmarket_emails.csv',
        'metric': 'Unsubscribe Rate',
        'date_col': 'Date'
    },
    
    # =========================================================================
    # NEWSLETTER SERIES - Podcasts (Page 12)
    # =========================================================================
    'Podcast Open Rate': {
        'image': 'newsletter_series_weekly/podcast_open_rate.png',
        'csv': 'newsletter_series/04_podcast_emails.csv',
        'metric': 'Open Rate',
        'date_col': 'Date'
    },
    'Podcast Verified CTR': {
        'image': 'newsletter_series_weekly/podcast_verified_ctr.png',
        'csv': 'newsletter_series/04_podcast_emails.csv',
        'metric': 'Verified Click-Through Rate',
        'date_col': 'Date'
    },
    'Podcast Unsubscribe Rate': {
        'image': 'newsletter_series_weekly/podcast_unsubscribe_rate.png',
        'csv': 'newsletter_series/04_podcast_emails.csv',
        'metric': 'Unsubscribe Rate',
        'date_col': 'Date'
    },
    
    # =========================================================================
    # NEWSLETTER SERIES - Case Study (Page 13)
    # =========================================================================
    'Case Study Open Rate': {
        'image': 'newsletter_series_weekly/case_study_open_rate.png',
        'csv': 'newsletter_series/01_case_study_emails.csv',
        'metric': 'Open Rate',
        'date_col': 'Date'
    },
    'Case Study Verified CTR': {
        'image': 'newsletter_series_weekly/case_study_verified_ctr.png',
        'csv': 'newsletter_series/01_case_study_emails.csv',
        'metric': 'Verified Click-Through Rate',
        'date_col': 'Date'
    },
    'Case Study Unsubscribe Rate': {
        'image': 'newsletter_series_weekly/case_study_unsubscribe_rate.png',
        'csv': 'newsletter_series/01_case_study_emails.csv',
        'metric': 'Unsubscribe Rate',
        'date_col': 'Date'
    },
    
    # =========================================================================
    # SALES METRICS - Lead Time (Page 16)
    # =========================================================================
    'Average Lead Time': {
        'image': 'discovery_intro_blended_weekly/weekly_avg_lead_time_completed.png',
        'csv': 'discovery_intro/weekly_discovery_intro_blended.csv',
        'metric': 'Avg Lead Time (Days)(Completed)',
        'date_col': 'Date Range'
    },
    
    # =========================================================================
    # SALES METRICS - Deal Upgrade (Page 17)
    # =========================================================================
    'Deal Upgrade Visits': {
        'image': 'deal_upgrade_weekly/weekly_visits.png',
        'csv': 'deal_upgrade/weekly_deal_upgrade.csv',
        'metric': 'Visits',
        'date_col': 'Date Range'
    },
    'Deal Upgrade CVR': {
        'image': 'deal_upgrade_weekly/weekly_conversion_rate.png',
        'csv': 'deal_upgrade/weekly_deal_upgrade.csv',
        'metric': 'Conversion Rate',
        'date_col': 'Date Range'
    },
    
    # =========================================================================
    # SALES METRICS - Pro Site (Page 18)
    # =========================================================================
    'Pro Site Visits': {
        'image': 'pro_site_weekly/weekly_visits.png',
        'csv': 'pro_site/weekly_pro_site.csv',
        'metric': 'Visits',
        'date_col': 'Date Range'
    },
    'Pro Site CVR': {
        'image': 'pro_site_weekly/weekly_conversion_rate.png',
        'csv': 'pro_site/weekly_pro_site.csv',
        'metric': 'Conversion Rate',
        'date_col': 'Date Range'
    },
    
    # =========================================================================
    # SALES METRICS - Booked Calls (Page 19)
    # =========================================================================
    'Booked Calls - Closers (Discovery Call)': {
        'image': 'discovery_call_weekly/weekly_booked_calls_completed.png',
        'csv': 'discovery_intro/weekly_discovery_call.csv',
        'metric': 'Booked Calls (Completed)',
        'date_col': 'Date Range'
    },
    'Booked Calls - Setters (Intro Call)': {
        'image': 'intro_call_weekly/weekly_booked_calls_completed.png',
        'csv': 'discovery_intro/weekly_intro_call.csv',
        'metric': 'Booked Calls (Completed)',
        'date_col': 'Date Range'
    },
    
    # =========================================================================
    # SALES METRICS - Sales Tracker (Pages 20-23)
    # =========================================================================
    'Scheduled Calls': {
        'image': 'sales_tracker_weekly/weekly_sched_calls.png',
        'csv': 'sales_tracker/weekly_sales_tracker.csv',
        'metric': 'Sched. calls',
        'date_col': 'Date Range'
    },
    'Live Calls': {
        'image': 'sales_tracker_weekly/weekly_live_calls.png',
        'csv': 'sales_tracker/weekly_sales_tracker.csv',
        'metric': 'Live calls',
        'date_col': 'Date Range'
    },
    'Show Rate': {
        'image': 'sales_tracker_weekly/weekly_show_pct.png',
        'csv': 'sales_tracker/weekly_sales_tracker.csv',
        'metric': 'Show %',
        'date_col': 'Date Range'
    },
    'Offers': {
        'image': 'sales_tracker_weekly/weekly_offers.png',
        'csv': 'sales_tracker/weekly_sales_tracker.csv',
        'metric': 'Offers',
        'date_col': 'Date Range'
    },
    'Offer Rate': {
        'image': 'sales_tracker_weekly/weekly_offer_pct.png',
        'csv': 'sales_tracker/weekly_sales_tracker.csv',
        'metric': 'Offer %',
        'date_col': 'Date Range'
    },
    'Closes': {
        'image': 'sales_tracker_weekly/weekly_close_1.png',
        'csv': 'sales_tracker/weekly_sales_tracker.csv',
        'metric': 'Close 1',
        'date_col': 'Date Range'
    },
    'Offer to Close Rate': {
        'image': 'sales_tracker_weekly/weekly_offer_to_close_pct.png',
        'csv': 'sales_tracker/weekly_sales_tracker.csv',
        'metric': 'Offer to Close %',
        'date_col': 'Date Range'
    },
}

print(f"✓ Chart configuration loaded: {len(CHART_CONFIG)} charts mapped")

✓ Chart configuration loaded: 37 charts mapped


### Section 4: Utility Functions - Value Parsing and Formatting

In [19]:
# =============================================================================
# SECTION 4: UTILITY FUNCTIONS - VALUE PARSING AND FORMATTING
# =============================================================================
# Helper functions for parsing values, formatting output, and determining
# metric types (percentage vs absolute values)

def parse_reference_dates(ref_file):
    """
    Parse analysis_ref_date.csv to extract week date range.
    
    How it works:
    - Reads the CSV line by line
    - Extracts key-value pairs (e.g., "Week Start Date,11/30/2025")
    - Returns formatted date strings for filename and display
    
    Returns:
        dict with keys: week_start, week_end, date_range_filename, date_range_display
    """
    if not os.path.exists(ref_file):
        print(f"  ⚠ Reference file not found: {ref_file}")
        return None
    
    try:
        ref_df = pd.read_csv(ref_file, header=None)
        ref_dict = dict(zip(ref_df[0], ref_df[1]))
        
        week_start = ref_dict.get('Week Start Date', '')
        week_end = ref_dict.get('Week End Date', '')
        
        # Format for filename: MM-DD-YYYY
        start_parts = week_start.split('/')
        end_parts = week_end.split('/')
        
        if len(start_parts) == 3 and len(end_parts) == 3:
            date_range_filename = f"{start_parts[0]}-{start_parts[1]}-{start_parts[2]}_to_{end_parts[0]}-{end_parts[1]}-{end_parts[2]}"
            date_range_display = f"{week_start} to {week_end}"
        else:
            date_range_filename = "unknown_date_range"
            date_range_display = "Unknown Date Range"
        
        return {
            'week_start': week_start,
            'week_end': week_end,
            'date_range_filename': date_range_filename,
            'date_range_display': date_range_display
        }
    except Exception as e:
        print(f"  ⚠ Error parsing reference dates: {e}")
        return None


def parse_value(value):
    """
    Parse a value that might be percentage string, number, or other format.
    
    Examples:
    - "50%" -> 50.0 (keeps as percentage number)
    - "+12.35%" -> 12.35
    - "33,821" -> 33821.0
    - 100 -> 100.0
    
    Returns:
        float or None if unparseable
    """
    if pd.isna(value):
        return None
    
    if isinstance(value, (int, float)):
        return float(value)
    
    value_str = str(value).strip().lstrip('+')
    
    # Handle percentage - just remove the % sign, keep the number
    if value_str.endswith('%'):
        try:
            return float(value_str.rstrip('%'))
        except ValueError:
            return None
    
    # Handle regular number (remove commas)
    try:
        return float(value_str.replace(',', ''))
    except ValueError:
        return None


def format_value(value, is_percentage=False, decimals=2):
    """
    Format numeric value for display in report.
    
    Args:
        value: The numeric value
        is_percentage: If True, add % sign
        decimals: Decimal places to show
    """
    if value is None or pd.isna(value):
        return 'N/A'
    
    if is_percentage:
        return f"{value:.{decimals}f}%"
    else:
        if abs(value) >= 1000:
            return f"{value:,.{decimals}f}"
        return f"{value:.{decimals}f}"


def format_change(value, is_percentage=False):
    """
    Format a change value with + or - sign for WoW comparisons.
    """
    if value is None or pd.isna(value):
        return 'N/A'
    
    sign = '+' if value > 0 else ''
    if is_percentage:
        return f"{sign}{value:.2f}%"
    return f"{sign}{value:.2f}"


def is_percentage_metric(metric_name):
    """
    Determine if metric should be displayed as percentage.
    Checks for keywords: rate, cvr, %, pct, percentage
    """
    metric_lower = metric_name.lower()
    pct_keywords = ['rate', 'cvr', '%', 'pct', 'percentage']
    return any(kw in metric_lower for kw in pct_keywords)


print("✓ Value parsing and formatting functions loaded")

✓ Value parsing and formatting functions loaded


### Section 5: Utility Functions - Trend and Spike Detection

In [20]:
# =============================================================================
# SECTION 5: UTILITY FUNCTIONS - TREND AND SPIKE DETECTION
# =============================================================================
# Statistical functions for analyzing time series data
# Uses two methods for robust trend detection:
# 1. Linear Regression with p-value (for clean data)
# 2. First-half vs Second-half average comparison (for volatile data)

def calculate_trend(values, p_value_threshold=0.05, min_points=4, half_comparison_threshold=0.05):
    """
    Calculate trend direction using a HYBRID approach:
    
    Method 1: Linear Regression with p-value
    - If p-value ≤ threshold → use slope direction (statistically significant)
    
    Method 2: First-half vs Second-half average comparison (for volatile data)
    - If p-value > threshold (volatile), compare averages of first half vs second half
    - If second half avg is significantly higher → Rising
    - If second half avg is significantly lower → Falling
    - Otherwise → Stable
    
    Args:
        values: List of values (oldest to newest)
        p_value_threshold: Significance level for linear regression (default 0.05)
        min_points: Minimum data points required
        half_comparison_threshold: Minimum % difference between halves to detect trend (default 5%)
    
    Returns:
        str: 'rising', 'falling', 'flat', or 'insufficient data'
    """
    clean_values = [v for v in values if v is not None and not pd.isna(v)]
    
    if len(clean_values) < min_points:
        return 'insufficient data'
    
    n = len(clean_values)
    x = np.arange(n)
    y = np.array(clean_values)
    
    # Method 1: Linear Regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    
    # If p-value is significant, use linear regression result
    if p_value <= p_value_threshold:
        if slope > 0:
            return 'rising'
        elif slope < 0:
            return 'falling'
        else:
            return 'flat'
    
    # Method 2: First-half vs Second-half comparison (for volatile data)
    # Split data into two halves
    mid_point = n // 2
    first_half = clean_values[:mid_point]
    second_half = clean_values[mid_point:]
    
    if len(first_half) < 2 or len(second_half) < 2:
        return 'flat'
    
    first_half_avg = np.mean(first_half)
    second_half_avg = np.mean(second_half)
    
    # Calculate percentage change between halves
    if first_half_avg != 0:
        pct_change = (second_half_avg - first_half_avg) / abs(first_half_avg)
    else:
        pct_change = 0
    
    # Classify based on percentage change between halves
    if pct_change > half_comparison_threshold:
        return 'rising'
    elif pct_change < -half_comparison_threshold:
        return 'falling'
    else:
        return 'flat'


def calculate_multi_period_trends(values):
    """
    Calculate trends over multiple time horizons using hybrid approach.
    
    Different thresholds for different periods:
    - 6wk: 8% change between halves needed (short term = more noise tolerance)
    - 12wk: 6% change between halves needed
    - 24wk: 5% change between halves needed (long term = smaller changes are meaningful)
    """
    short_period = CONFIG['trend_short_term']    # 6
    medium_period = CONFIG['trend_medium_term']  # 12
    long_period = CONFIG['trend_long_term']      # 24
    
    trends = {
        'short': {'period': short_period, 'trend': 'insufficient data'},
        'medium': {'period': medium_period, 'trend': 'insufficient data'},
        'long': {'period': long_period, 'trend': 'insufficient data'},
    }
    
    num_points = len(values)
    
    # Short-term trend (need at least 6 points)
    if num_points >= short_period:
        trends['short']['trend'] = calculate_trend(
            values[-short_period:], 
            p_value_threshold=0.05, 
            min_points=4,
            half_comparison_threshold=0.08  # 8% for 6 weeks
        )
    
    # Medium-term trend (need at least 12 points)
    if num_points >= medium_period:
        trends['medium']['trend'] = calculate_trend(
            values[-medium_period:], 
            p_value_threshold=0.05, 
            min_points=6,
            half_comparison_threshold=0.06  # 6% for 12 weeks
        )
    
    # Long-term trend (need at least 24 points)
    if num_points >= long_period:
        trends['long']['trend'] = calculate_trend(
            values[-long_period:], 
            p_value_threshold=0.05, 
            min_points=12,
            half_comparison_threshold=0.05  # 5% for 24 weeks
        )
    
    return trends


def classify_vs_average(current, six_week_avg):
    """
    Classify current value relative to 6-week average.
    
    Returns:
        str: 'High CV vs 6Wk Avg' (>10% above), 'Low CV vs 6Wk Avg' (<10% below), or 'Normal'
    """
    if current is None or six_week_avg is None or six_week_avg == 0:
        return 'Normal'
    
    pct_diff = (current - six_week_avg) / abs(six_week_avg)
    
    if pct_diff > 0.10:
        return 'High CV vs 6Wk Avg'
    elif pct_diff < -0.10:
        return 'Low CV vs 6Wk Avg'
    return 'Normal'


print("✓ Trend detection functions loaded (hybrid: p-value + half comparison)")

✓ Trend detection functions loaded (hybrid: p-value + half comparison)


### Section 6: Data Loading Functions

In [21]:
# =============================================================================
# SECTION 6: DATA LOADING FUNCTIONS
# =============================================================================
# Functions to load CSV and Excel files, extract time series data
# Data is filtered to only include records up to the reporting week end date

# Global variable for reporting week end date filter
REPORTING_WEEK_END = None

def set_reporting_week_end(date_info):
    """
    Set the global reporting week end date used to filter data.
    Only data up to this date will be included in analysis.
    """
    global REPORTING_WEEK_END
    week_end_str = date_info.get('week_end', '')
    
    formats = ['%m/%d/%Y', '%Y-%m-%d', '%m/%d/%y']
    for fmt in formats:
        try:
            REPORTING_WEEK_END = datetime.strptime(week_end_str, fmt)
            print(f"  Data filter: Only including data up to {REPORTING_WEEK_END.strftime('%m/%d/%Y')}")
            return
        except ValueError:
            continue
    
    print(f"  ⚠ Could not parse week end date: {week_end_str}")
    REPORTING_WEEK_END = None


def load_csv_data(csv_path):
    """Load and clean a CSV file, removing empty rows."""
    if not os.path.exists(csv_path):
        return None
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(how='all')
        return df
    except Exception as e:
        print(f"  ⚠ Error loading CSV: {e}")
        return None


def load_excel_metrics(xlsx_path):
    """Load metrics table from Excel file, removing empty rows and columns."""
    if not os.path.exists(xlsx_path):
        return None
    try:
        df = pd.read_excel(xlsx_path)
        df = df.dropna(how='all').dropna(axis=1, how='all')
        return df
    except Exception as e:
        print(f"  ⚠ Error loading Excel: {e}")
        return None


def parse_date_for_sorting(date_str):
    """
    Parse date string to datetime object for sorting and filtering.
    For date ranges like "11/30/2025 - 12/06/2025", extracts the END date.
    """
    if pd.isna(date_str):
        return None
    
    date_str = str(date_str).strip()
    
    # Extract END date from date ranges
    if ' - ' in date_str:
        date_str = date_str.split(' - ')[-1].strip()
    
    # Skip non-date strings
    if date_str.lower().startswith('week') or len(date_str) < 6:
        return None
    
    formats = ['%m/%d/%Y', '%Y-%m-%d', '%m/%d/%y', '%d/%m/%Y', '%m-%d-%Y']
    
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    
    return None


def extract_metric_series(df, metric_col, date_col):
    """
    Extract time series for a metric column from dataframe.
    
    Filtering:
    - Only includes data where date <= REPORTING_WEEK_END
    - Data beyond the reporting week is ignored
    
    Sorting:
    - Returns data sorted chronologically (oldest first, newest last)
    - values[-1] = most recent week (the reporting week)
    - values[-2] = previous week
    
    Returns:
        List of (date_string, value) tuples sorted oldest to newest
    """
    global REPORTING_WEEK_END
    
    if df is None or metric_col not in df.columns:
        return []
    
    raw_data = []
    
    # Handle two-column date format: [Start Date, End Date]
    if isinstance(date_col, list) and len(date_col) == 2:
        start_col, end_col = date_col
        if start_col in df.columns and end_col in df.columns:
            for idx, row in df.iterrows():
                date_display = f"{row[start_col]} - {row[end_col]}"
                value = parse_value(row[metric_col])
                sort_date = parse_date_for_sorting(str(row[end_col]))
                if value is not None:
                    raw_data.append({
                        'date_display': date_display,
                        'value': value,
                        'sort_date': sort_date
                    })
    # Handle single date column
    elif date_col in df.columns:
        for idx, row in df.iterrows():
            date_display = str(row[date_col])
            value = parse_value(row[metric_col])
            sort_date = parse_date_for_sorting(date_display)
            if value is not None:
                raw_data.append({
                    'date_display': date_display,
                    'value': value,
                    'sort_date': sort_date
                })
    
    if not raw_data:
        return []
    
    # Separate records with and without parseable dates
    with_dates = [d for d in raw_data if d['sort_date'] is not None]
    without_dates = [d for d in raw_data if d['sort_date'] is None]
    
    # Filter: only include data up to reporting week end date
    if REPORTING_WEEK_END is not None:
        with_dates = [d for d in with_dates if d['sort_date'] <= REPORTING_WEEK_END]
    
    # Sort chronologically: oldest first, newest last
    with_dates.sort(key=lambda x: x['sort_date'])
    
    # Build result list
    result = [(d['date_display'], d['value']) for d in with_dates]
    result.extend([(d['date_display'], d['value']) for d in without_dates])
    
    return result


print("✓ Data loading functions loaded")

✓ Data loading functions loaded


### Section 7: Chart Insight Generation

In [22]:
# =============================================================================
# SECTION 7: CHART INSIGHT GENERATION
# =============================================================================
# Functions to calculate insights from chart data with multi-period trends
# Handles decimal-to-percent conversion for specific files/metrics

# Define which CSV files and metrics store values as decimals (need *100)
# Only general_newsletter/weekly_table.csv stores these metrics as decimals
DECIMAL_METRICS = {
    'general_newsletter/weekly_table.csv': ['Growth Rate', 'Newsletter CVR %', 'Leak Rate', 'Unsub Rate']
}

def needs_decimal_conversion(csv_path, metric_name):
    """
    Check if this metric needs decimal-to-percent conversion.
    
    Only general_newsletter/weekly_table.csv stores certain metrics as decimals.
    All other CSV files are already in percentage form.
    """
    for csv_key, metrics in DECIMAL_METRICS.items():
        if csv_key in csv_path:
            if any(m.lower() == metric_name.lower() for m in metrics):
                return True
    return False


def get_chart_data_insights(chart_name, config):
    """
    Load chart data and calculate all insights for the reporting week.
    
    Handles:
    - Absolute value metrics (converts negative to positive)
    - Inverted metrics (where lower is better)
    - Decimal to percent conversion (only for specific files/metrics)
    - Multi-period trend analysis with Linear Regression
    """
    csv_path = os.path.join(OUTPUTS_PATH, config['csv'])
    df = load_csv_data(csv_path)
    
    insights = {
        'chart_name': chart_name,
        'csv_source': config['csv'],
        'metric': config['metric'],
        'data_available': False,
        'data_points': 0
    }
    
    if df is None:
        insights['error'] = 'CSV file not found'
        return insights
    
    # Extract filtered and sorted time series
    series = extract_metric_series(df, config['metric'], config['date_col'])
    
    if len(series) < 2:
        insights['error'] = f'Not enough data points ({len(series)})'
        insights['data_points'] = len(series)
        return insights
    
    insights['data_available'] = True
    insights['data_points'] = len(series)
    
    # Extract values (series is now 2-tuple: (date, value))
    values = [v for _, v in series]
    
    # Check if this metric should use absolute values
    is_absolute_metric = any(abs_m.lower() in config['metric'].lower() 
                            for abs_m in ABSOLUTE_VALUE_METRICS)
    
    if is_absolute_metric:
        values = [abs(v) for v in values]
    
    # Current = last value (reporting week)
    # Previous = second-to-last value (week before)
    current = values[-1]
    previous = values[-2]
    
    # Check if this specific file/metric needs decimal conversion
    is_pct_metric = is_percentage_metric(config['metric'])
    needs_conversion = needs_decimal_conversion(config['csv'], config['metric'])
    
    if is_pct_metric and needs_conversion:
        # This metric is stored as decimal (e.g., 0.20 for 20%), convert to percentage
        current_display = current * 100
        previous_display = previous * 100
    else:
        # Already in percentage form or not a percentage metric
        current_display = current
        previous_display = previous
    
    insights['current_value'] = current_display
    insights['previous_value'] = previous_display
    insights['current_date'] = series[-1][0]
    insights['previous_date'] = series[-2][0]
    insights['is_absolute_metric'] = is_absolute_metric
    insights['is_pct_metric'] = is_pct_metric
    insights['needs_conversion'] = needs_conversion
    
    # Week-over-week change calculation
    if previous != 0:
        insights['wow_change_pct'] = (current - previous) / abs(previous)
        insights['wow_change_abs'] = current_display - previous_display
    else:
        insights['wow_change_pct'] = None
        insights['wow_change_abs'] = current_display - previous_display
    
    # 6-week statistics (convert to display values if needed)
    last_6 = values[-6:] if len(values) >= 6 else values
    if is_pct_metric and needs_conversion:
        last_6_display = [v * 100 for v in last_6]
    else:
        last_6_display = last_6
    insights['six_week_avg'] = np.mean(last_6_display)
    insights['six_week_std'] = np.std(last_6_display) if len(last_6_display) >= 2 else 0
    
    # Classification vs 6-week average
    insights['vs_average'] = classify_vs_average(current_display, insights['six_week_avg'])
    
    # Multi-period trend analysis
    insights['trends'] = calculate_multi_period_trends(values)
    
    return insights


def format_trend_with_arrow(trend):
    """Convert trend string to display format with arrow."""
    if trend == 'rising':
        return '↑ Rising'
    elif trend == 'falling':
        return '↓ Falling'
    elif trend == 'flat':
        return '→ Stable'
    else:
        return None  # Return None for insufficient data


def generate_chart_insight_text(insights, metric_name):
    """
    Generate human-readable insight paragraph from calculated insights.
    
    Output format:
    - Line 1: Current value with WoW change and favorable/unfavorable label
    - Line 2: 6-week average and classification
    - Line 3: Multi-period trends
    
    Features:
    - Zero change shows "no change from previous week"
    - Percentages always show 2 decimal places (XX.XX%)
    """
    if not insights.get('data_available', False):
        return f"⚠ {insights.get('error', 'Data unavailable')}"
    
    is_pct = insights.get('is_pct_metric', is_percentage_metric(metric_name))
    is_inverted = any(inv.lower() in metric_name.lower() for inv in INVERTED_METRICS)
    
    current = insights['current_value']
    previous = insights['previous_value']
    wow_pct = insights.get('wow_change_pct')
    six_avg = insights['six_week_avg']
    
    lines = []
    
    # Format values based on metric type (always 2 decimal places for percentages)
    if is_pct:
        curr_str = f"{current:.2f}%"
        prev_str = f"{previous:.2f}%"
        avg_str = f"{six_avg:.2f}%"
    else:
        curr_str = f"{current:,.0f}" if current >= 100 else f"{current:.2f}"
        prev_str = f"{previous:,.0f}" if abs(previous) >= 100 else f"{previous:.2f}"
        avg_str = f"{six_avg:,.0f}" if six_avg >= 100 else f"{six_avg:.2f}"
    
    # Line 1: Current value and WoW change
    if wow_pct is not None:
        # Check for zero or near-zero change
        if abs(wow_pct) < 0.001:  # Less than 0.1% change
            lines.append(f"Current week: {curr_str} (no change from previous week)")
        else:
            direction = "up" if wow_pct > 0 else "down"
            if is_inverted:
                # For inverted metrics: down = good, up = bad
                quality = "(favorable)" if wow_pct < 0 else "(unfavorable)"
            else:
                # For normal metrics: up = good, down = bad
                quality = "(favorable)" if wow_pct > 0 else "(unfavorable)"
            lines.append(f"Current week: {curr_str} ({direction} {abs(wow_pct)*100:.2f}% WoW from {prev_str}) {quality}")
    else:
        lines.append(f"Current week: {curr_str} (previous: {prev_str})")
    
    # Line 2: 6-week average comparison
    lines.append(f"6-week average: {avg_str} — classification: {insights['vs_average']}")
    
    # Line 3: Multi-period trends
    trends = insights.get('trends', {})
    trend_parts = []
    
    # Short-term (6 weeks)
    short_trend = trends.get('short', {})
    short_formatted = format_trend_with_arrow(short_trend.get('trend'))
    if short_formatted:
        trend_parts.append(f"6wk {short_formatted}")
    
    # Medium-term (12 weeks)
    medium_trend = trends.get('medium', {})
    medium_formatted = format_trend_with_arrow(medium_trend.get('trend'))
    if medium_formatted:
        trend_parts.append(f"12wk {medium_formatted}")
    
    # Long-term (24 weeks)
    long_trend = trends.get('long', {})
    long_formatted = format_trend_with_arrow(long_trend.get('trend'))
    if long_formatted:
        trend_parts.append(f"24wk {long_formatted}")
    
    # Build trend line
    if trend_parts:
        lines.append("Trends: " + " | ".join(trend_parts))
    else:
        lines.append(f"Trends: Insufficient data ({insights['data_points']} weeks available)")
    
    return "\n".join(lines)


print("✓ Chart insight generation functions loaded (with decimal conversion for weekly_table.csv)")

✓ Chart insight generation functions loaded (with decimal conversion for weekly_table.csv)


### Section 8: Section Summary Generation

In [23]:
# =============================================================================
# SECTION 8: SECTION SUMMARY GENERATION
# =============================================================================
# Functions to generate section summaries and executive summary

def generate_section_summary(metrics_df, section_type='newsletter'):
    """
    Generate section summary highlighting top improvements and concerns.
    
    How it works:
    1. Parse WoW column to get percentage changes
    2. Categorize each metric as improvement or concern
       - For normal metrics: positive change = improvement
       - For inverted metrics: negative change = improvement
    3. Sort by magnitude and take top 3 of each
    
    Args:
        metrics_df: DataFrame with Metric and WoW columns
        section_type: 'newsletter' or 'sales'
    
    Returns:
        dict with 'improvements', 'concerns', and 'summary' text
    """
    if metrics_df is None or 'Metric' not in metrics_df.columns:
        return {'improvements': [], 'concerns': [], 'summary': 'No data available'}
    
    # Find WoW/MoM column
    wow_col = None
    for col in metrics_df.columns:
        if 'wow' in col.lower() or 'mom' in col.lower() or 'mtd' in col.lower():
            wow_col = col
            break
    
    if wow_col is None:
        return {'improvements': [], 'concerns': [], 'summary': 'No change column found'}
    
    improvements = []
    concerns = []
    
    for _, row in metrics_df.iterrows():
        metric = row['Metric']
        wow_val = parse_value(row[wow_col])
        
        if wow_val is None:
            continue
        
        is_inverted = any(inv.lower() in metric.lower() for inv in INVERTED_METRICS)
        
        # Categorize based on metric type
        if is_inverted:
            # Inverted: negative change is good
            if wow_val < -0.05:
                improvements.append((metric, wow_val, 'decreased'))
            elif wow_val > 0.05:
                concerns.append((metric, wow_val, 'increased'))
        else:
            # Normal: positive change is good
            if wow_val > 0.05:
                improvements.append((metric, wow_val, 'increased'))
            elif wow_val < -0.05:
                concerns.append((metric, wow_val, 'decreased'))
    
    # Sort by magnitude and take top 3
    improvements.sort(key=lambda x: abs(x[1]), reverse=True)
    concerns.sort(key=lambda x: abs(x[1]), reverse=True)
    
    top_improvements = improvements[:3]
    top_concerns = concerns[:3]
    
    # Build summary text
    summary_parts = []
    
    if top_improvements:
        summary_parts.append("Key Improvements:")
        for metric, change, direction in top_improvements:
            summary_parts.append(f"  • {metric} {direction} by {abs(change)*100:.1f}%")
    
    if top_concerns:
        summary_parts.append("\nAreas of Concern:")
        for metric, change, direction in top_concerns:
            summary_parts.append(f"  • {metric} {direction} by {abs(change)*100:.1f}%")
    
    return {
        'improvements': top_improvements,
        'concerns': top_concerns,
        'summary': '\n'.join(summary_parts) if summary_parts else 'No significant changes detected'
    }


def generate_executive_summary(newsletter_df, sales_df):
    """
    Generate executive summary combining newsletter and sales highlights.
    
    Returns:
        str: Combined summary text for both sections
    """
    newsletter_summary = generate_section_summary(newsletter_df, 'newsletter')
    sales_summary = generate_section_summary(sales_df, 'sales')
    
    summary_text = []
    summary_text.append("NEWSLETTER PERFORMANCE")
    summary_text.append(newsletter_summary['summary'])
    summary_text.append("")
    summary_text.append("SALES PERFORMANCE")
    summary_text.append(sales_summary['summary'])
    
    return '\n'.join(summary_text)


print("✓ Section summary generation functions loaded")

✓ Section summary generation functions loaded


### Section 9: Document Formatting Functions

In [24]:
# =============================================================================
# SECTION 9: DOCUMENT FORMATTING FUNCTIONS
# =============================================================================
# Functions for document layout, tables, and page management
# Now includes color formatting for ALL tables (weekly AND monthly)
# Now includes source file display below tables

from docx.oxml import OxmlElement
from docx.oxml.ns import qn


def add_page_break(doc):
    """
    Add a page break to the document.
    Creates a new paragraph with a page break.
    Use insert_page_break_before() when possible to avoid blank pages.
    """
    doc.add_page_break()


def insert_page_break_before(paragraph):
    """
    Insert a page break at the START of an existing paragraph.
    This prevents blank pages by attaching the break to content.
    """
    run = paragraph.runs[0] if paragraph.runs else paragraph.add_run()
    br = OxmlElement('w:br')
    br.set(qn('w:type'), 'page')
    run._r.insert(0, br)


def set_cell_shading(cell, color_hex):
    """Apply background shading to a table cell."""
    shading = parse_xml(f'<w:shd {nsdecls("w")} w:fill="{color_hex}"/>')
    cell._tc.get_or_add_tcPr().append(shading)


def add_formatted_table(doc, df, title=None, color_wow=True, source_file=None):
    """
    Add a formatted data table to the document.
    Color codes the WoW/MoM column based on positive/negative values.
    Works for BOTH weekly and monthly tables.
    
    Args:
        doc: Document object
        df: DataFrame with table data
        title: Optional title above table
        color_wow: Whether to color code the WoW/MoM column
        source_file: Optional source file path to display below table
    """
    if title:
        p = doc.add_paragraph()
        run = p.add_run(title)
        run.bold = True
        run.font.size = Pt(14)
        p.paragraph_format.space_after = Pt(6)
    
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = 'Table Grid'
    
    # Header row
    header_cells = table.rows[0].cells
    for i, col_name in enumerate(df.columns):
        header_cells[i].text = str(col_name)
        header_cells[i].paragraphs[0].runs[0].bold = True
        header_cells[i].paragraphs[0].runs[0].font.size = Pt(11)
        set_cell_shading(header_cells[i], "D9E2F3")
    
    # Find the WoW/MoM column index (usually last or second-to-last)
    wow_col_idx = None
    for i, col in enumerate(df.columns):
        col_lower = str(col).lower()
        if 'wow' in col_lower or 'mom' in col_lower or col_lower in ['wow', 'mom', 'mom/mtd']:
            wow_col_idx = i
            break
    
    # If not found by name, assume it's the last column with % values
    if wow_col_idx is None:
        wow_col_idx = len(df.columns) - 1
    
    # Data rows
    for _, row in df.iterrows():
        row_cells = table.add_row().cells
        for i, value in enumerate(row):
            cell = row_cells[i]
            cell.text = str(value) if pd.notna(value) else ""
            
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(11)
            
            # Color code WoW/MoM column
            if color_wow and i == wow_col_idx:
                try:
                    val_str = str(value).replace('%', '').replace('+', '').strip()
                    val_num = float(val_str)
                    metric_name = str(row.iloc[0]).lower() if len(row) > 0 else ""
                    
                    is_inverted = any(inv.lower() in metric_name for inv in INVERTED_METRICS)
                    
                    if is_inverted:
                        # For inverted metrics: negative = good (green), positive = bad (red)
                        if val_num < 0:
                            set_cell_shading(cell, "C6EFCE")  # Green
                        elif val_num > 0:
                            set_cell_shading(cell, "FFC7CE")  # Red
                    else:
                        # For normal metrics: positive = good (green), negative = bad (red)
                        if val_num > 0:
                            set_cell_shading(cell, "C6EFCE")  # Green
                        elif val_num < 0:
                            set_cell_shading(cell, "FFC7CE")  # Red
                except (ValueError, TypeError):
                    pass
    
    # Add source file below table (dark gray, small font)
    if source_file:
        p = doc.add_paragraph()
        run = p.add_run(f"Source: {source_file}")
        run.font.size = Pt(7)
        run.font.color.rgb = RGBColor(100, 100, 100)  # Dark gray
        p.paragraph_format.space_before = Pt(2)
        p.paragraph_format.space_after = Pt(6)


def add_page_numbers(doc):
    """Add page numbers to document footer."""
    for section in doc.sections:
        footer = section.footer
        footer.is_linked_to_previous = False
        
        p = footer.paragraphs[0] if footer.paragraphs else footer.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        run1 = p.add_run("Page ")
        run1.font.size = Pt(10)
        
        fldChar1 = OxmlElement('w:fldChar')
        fldChar1.set(qn('w:fldCharType'), 'begin')
        instrText1 = OxmlElement('w:instrText')
        instrText1.text = "PAGE"
        fldChar2 = OxmlElement('w:fldChar')
        fldChar2.set(qn('w:fldCharType'), 'separate')
        fldChar3 = OxmlElement('w:fldChar')
        fldChar3.set(qn('w:fldCharType'), 'end')
        
        run2 = p.add_run()
        run2._r.append(fldChar1)
        run2._r.append(instrText1)
        run2._r.append(fldChar2)
        run2._r.append(fldChar3)
        run2.font.size = Pt(10)
        
        run3 = p.add_run(" of ")
        run3.font.size = Pt(10)
        
        fldChar4 = OxmlElement('w:fldChar')
        fldChar4.set(qn('w:fldCharType'), 'begin')
        instrText2 = OxmlElement('w:instrText')
        instrText2.text = "NUMPAGES"
        fldChar5 = OxmlElement('w:fldChar')
        fldChar5.set(qn('w:fldCharType'), 'separate')
        fldChar6 = OxmlElement('w:fldChar')
        fldChar6.set(qn('w:fldCharType'), 'end')
        
        run4 = p.add_run()
        run4._r.append(fldChar4)
        run4._r.append(instrText2)
        run4._r.append(fldChar5)
        run4._r.append(fldChar6)
        run4.font.size = Pt(10)


print("✓ Document formatting functions loaded (with table source info)")

✓ Document formatting functions loaded (with table source info)


### Section 10: Chart with Insights Function

In [25]:
# =============================================================================
# SECTION 10: CHART WITH INSIGHTS FUNCTION
# =============================================================================
# Adds chart image and insights to the document
# Charts are sized to fit 2 per page with their insights
# Now includes image filename and data source below the chart

def add_chart_with_insights(doc, chart_name, config, source_log, page_break_before=False):
    """
    Add a chart image with auto-generated insights to the document.
    Chart + insights are kept compact to fit 2 per page.
    
    Now includes:
    - Image filename with full path (e.g., visualizations/discovery_intro_blended_weekly/weekly_avg_lead_time_completed.png)
    - Data source path (e.g., outputs/discovery_intro/weekly_discovery_intro_blended.csv)
    """
    # Chart title
    p = doc.add_paragraph()
    run = p.add_run(chart_name)
    run.bold = True
    run.font.size = Pt(11)
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(2)
    
    # Add page break to this paragraph if requested
    if page_break_before:
        insert_page_break_before(p)
    
    # Chart image - smaller size
    image_path = os.path.join(VISUALIZATIONS_PATH, config['image'])
    
    if os.path.exists(image_path):
        pic = doc.add_picture(image_path, width=Inches(CONFIG['chart_width_inches']))
        # Reduce space after image
        last_para = doc.paragraphs[-1]
        last_para.paragraph_format.space_after = Pt(1)
        source_log.append(f"  Chart: {config['image']}")
    else:
        doc.add_paragraph(f"[Chart not found: {config['image']}]")
        source_log.append(f"  Chart: {config['image']} (NOT FOUND)")
    
    # Add image path and data source below chart (dark gray, small font)
    image_source = f"visualizations/{config['image']}"
    data_source = f"outputs/{config['csv']}"
    
    p = doc.add_paragraph()
    # Image source (full path)
    run = p.add_run(f"{image_source}\n")
    run.font.size = Pt(7)
    run.font.color.rgb = RGBColor(100, 100, 100)  # Dark gray
    # Data source
    run = p.add_run(f"Source: {data_source}")
    run.font.size = Pt(7)
    run.font.color.rgb = RGBColor(100, 100, 100)  # Dark gray
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(2)
    
    # Generate insights
    insights = get_chart_data_insights(chart_name, config)
    insight_text = generate_chart_insight_text(insights, config['metric'])
    
    # Insight label
    p = doc.add_paragraph()
    run = p.add_run("Insights:")
    run.bold = True
    run.font.size = Pt(9)
    p.paragraph_format.space_before = Pt(2)
    p.paragraph_format.space_after = Pt(1)
    
    # Insight text - compact
    p = doc.add_paragraph(insight_text)
    p.paragraph_format.left_indent = Inches(0.15)
    p.paragraph_format.space_after = Pt(8)
    for run in p.runs:
        run.font.size = Pt(9)
    
    source_log.append(f"  Data: {config['csv']} → {config['metric']}")
    
    return True


print("✓ Chart with insights function loaded (with source info)")


✓ Chart with insights function loaded (with source info)


### Section 11: Cover Page and Executive Summary

In [26]:
# =============================================================================
# SECTION 11: COVER PAGE AND EXECUTIVE SUMMARY
# =============================================================================
# Creates the report cover page with title, dates, and table of contents
# Creates executive summary on page 2

from docx.oxml import OxmlElement
from docx.oxml.ns import qn

def add_table_of_contents(doc):
    """
    Add a Table of Contents field that Word will generate.
    User must right-click and select "Update Field" in Word to populate it.
    """
    paragraph = doc.add_paragraph()
    run = paragraph.add_run()
    
    fldChar1 = OxmlElement('w:fldChar')
    fldChar1.set(qn('w:fldCharType'), 'begin')
    
    instrText = OxmlElement('w:instrText')
    instrText.set(qn('xml:space'), 'preserve')
    instrText.text = 'TOC \\o "1-2" \\h \\z \\u'
    
    fldChar2 = OxmlElement('w:fldChar')
    fldChar2.set(qn('w:fldCharType'), 'separate')
    
    fldChar3 = OxmlElement('w:fldChar')
    fldChar3.set(qn('w:fldCharType'), 'end')
    
    run._r.append(fldChar1)
    run._r.append(instrText)
    run._r.append(fldChar2)
    
    placeholder_run = paragraph.add_run("Right-click and select 'Update Field' to generate Table of Contents")
    placeholder_run.italic = True
    placeholder_run.font.color.rgb = RGBColor(128, 128, 128)
    
    run2 = paragraph.add_run()
    run2._r.append(fldChar3)
    
    return doc


def create_cover_page(doc, date_info):
    """
    Create cover page with title, reporting week dates, and table of contents.
    """
    # Title
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = title.add_run("Weekly Business Review (WBR)")
    run.bold = True
    run.font.size = Pt(28)
    
    doc.add_paragraph()
    
    # Reporting week
    subtitle = doc.add_paragraph()
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = subtitle.add_run(f"Reporting Week: {date_info['date_range_display']}")
    run.font.size = Pt(16)
    
    doc.add_paragraph()
    
    # Generation timestamp
    gen_date = doc.add_paragraph()
    gen_date.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = gen_date.add_run(f"Report Generated: {datetime.now().strftime('%B %d, %Y at %I:%M %p')}")
    run.font.size = Pt(11)
    run.italic = True
    
    doc.add_paragraph()
    doc.add_paragraph()
    
    # Table of Contents header
    toc_header = doc.add_paragraph()
    run = toc_header.add_run("Table of Contents")
    run.bold = True
    run.font.size = Pt(14)
    
    # Add TOC field
    add_table_of_contents(doc)
    
    return doc


def create_executive_summary(doc, newsletter_df, sales_df):
    """
    Create Executive Summary on page 2 with key improvements and concerns.
    """
    add_page_break(doc)
    
    doc.add_heading("Executive Summary", level=1)
    
    # Generate summary from metrics tables
    exec_summary = generate_executive_summary(newsletter_df, sales_df)
    
    for line in exec_summary.split('\n'):
        if line.strip():
            p = doc.add_paragraph(line)
            p.paragraph_format.space_after = Pt(3)
            # Bold section headers
            if line.isupper() or (line.endswith(':') and not line.startswith(' ')):
                for run in p.runs:
                    run.bold = True
    
    return doc


print("✓ Cover page and executive summary functions loaded")

✓ Cover page and executive summary functions loaded


### Section 12: Newsletter Section Generation

In [27]:
# =============================================================================
# SECTION 12: NEWSLETTER SECTION GENERATION
# =============================================================================
# Now uses add_formatted_table for BOTH weekly and monthly tables
# This ensures color formatting is applied to MoM tables
# Now includes source file display below tables

def create_newsletter_section(doc, source_log):
    """Create Newsletter Metrics section."""
    
    # Section 1 header (with page break before it)
    h = doc.add_heading("Section 1: Newsletter Metrics", level=1)
    insert_page_break_before(h)
    
    # Weekly metrics table
    weekly_path = os.path.join(OUTPUTS_PATH, 'newsletter_analysis/weekly_newsletter_metrics.xlsx')
    weekly_df = load_excel_metrics(weekly_path)
    if weekly_df is not None:
        source_log.append(f"\n--- Newsletter Weekly Table ---")
        add_formatted_table(
            doc, weekly_df, 
            title="Core Metrics — Weekly Overview (WoW)", 
            color_wow=True,
            source_file="outputs/newsletter_analysis/weekly_newsletter_metrics.xlsx"
        )
    
    # Monthly metrics table (with page break) - NOW USES add_formatted_table for color coding
    monthly_path = os.path.join(OUTPUTS_PATH, 'newsletter_analysis/monthly_newsletter_metrics.xlsx')
    monthly_df = load_excel_metrics(monthly_path)
    if monthly_df is not None:
        source_log.append(f"\n--- Newsletter Monthly Table ---")
        # Create title paragraph with page break
        p = doc.add_paragraph()
        run = p.add_run("Core Metrics — Monthly Overview (MoM)")
        run.bold = True
        run.font.size = Pt(14)
        p.paragraph_format.space_after = Pt(6)
        insert_page_break_before(p)
        
        # Add table with color formatting (no title since we added it above)
        add_formatted_table(
            doc, monthly_df, 
            title=None, 
            color_wow=True,
            source_file="outputs/newsletter_analysis/monthly_newsletter_metrics.xlsx"
        )
    
    # LANDING PAGE FUNNEL
    h = doc.add_heading("Landing Page Funnel", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Landing Page Funnel Charts ---")
    add_chart_with_insights(doc, 'Newsletter Visits', CHART_CONFIG['Newsletter Visits'], source_log)
    add_chart_with_insights(doc, 'Newsletter Visit Duration', CHART_CONFIG['Newsletter Visit Duration'], source_log)
    add_chart_with_insights(doc, 'Newsletter Bounce Rate', CHART_CONFIG['Newsletter Bounce Rate'], source_log, page_break_before=True)
    add_chart_with_insights(doc, 'Newsletter CVR %', CHART_CONFIG['Newsletter CVR %'], source_log)
    
    # ENGAGEMENT METRICS
    h = doc.add_heading("Engagement Metrics", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Engagement Charts ---")
    add_chart_with_insights(doc, 'Blended Open Rate', CHART_CONFIG['Blended Open Rate'], source_log)
    add_chart_with_insights(doc, 'Blended Verified Click-Through Rate', CHART_CONFIG['Blended Verified Click-Through Rate'], source_log)
    add_chart_with_insights(doc, 'Blended Unsub Rate', CHART_CONFIG['Blended Unsub Rate'], source_log, page_break_before=True)
    
    # GROWTH & CHURN
    h = doc.add_heading("Growth & Churn", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Growth & Churn Charts ---")
    add_chart_with_insights(doc, 'Growth Rate', CHART_CONFIG['Growth Rate'], source_log)
    add_chart_with_insights(doc, 'New Subscribers', CHART_CONFIG['New Subscribers'], source_log)
    add_chart_with_insights(doc, 'Unsubscribes', CHART_CONFIG['Unsubscribes'], source_log, page_break_before=True)
    add_chart_with_insights(doc, 'Leak Rate', CHART_CONFIG['Leak Rate'], source_log)
    
    # NEW DEALS
    h = doc.add_heading("Newsletter Series: New Deals", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- New Deals Series ---")
    add_chart_with_insights(doc, 'Deal Open Rate', CHART_CONFIG['Deal Open Rate'], source_log)
    add_chart_with_insights(doc, 'Deal Verified CTR', CHART_CONFIG['Deal Verified CTR'], source_log)
    add_chart_with_insights(doc, 'Deal Unsubscribe Rate', CHART_CONFIG['Deal Unsubscribe Rate'], source_log, page_break_before=True)
    
    # OFF-MARKET
    h = doc.add_heading("Newsletter Series: Off-Market", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Off-Market Series ---")
    add_chart_with_insights(doc, 'Off-Market Open Rate', CHART_CONFIG['Off-Market Open Rate'], source_log)
    add_chart_with_insights(doc, 'Off-Market Verified CTR', CHART_CONFIG['Off-Market Verified CTR'], source_log)
    add_chart_with_insights(doc, 'Off-Market Unsubscribe Rate', CHART_CONFIG['Off-Market Unsubscribe Rate'], source_log, page_break_before=True)
    
    # PODCASTS
    h = doc.add_heading("Newsletter Series: Podcasts", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Podcast Series ---")
    add_chart_with_insights(doc, 'Podcast Open Rate', CHART_CONFIG['Podcast Open Rate'], source_log)
    add_chart_with_insights(doc, 'Podcast Verified CTR', CHART_CONFIG['Podcast Verified CTR'], source_log)
    add_chart_with_insights(doc, 'Podcast Unsubscribe Rate', CHART_CONFIG['Podcast Unsubscribe Rate'], source_log, page_break_before=True)
    
    # CASE STUDY
    h = doc.add_heading("Newsletter Series: Case Study", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Case Study Series ---")
    add_chart_with_insights(doc, 'Case Study Open Rate', CHART_CONFIG['Case Study Open Rate'], source_log)
    add_chart_with_insights(doc, 'Case Study Verified CTR', CHART_CONFIG['Case Study Verified CTR'], source_log)
    add_chart_with_insights(doc, 'Case Study Unsubscribe Rate', CHART_CONFIG['Case Study Unsubscribe Rate'], source_log, page_break_before=True)
    
    return doc


print("✓ Newsletter section function loaded (with table source info)")

✓ Newsletter section function loaded (with table source info)


### Section 13: Sales Section Generation

In [28]:
# =============================================================================
# SECTION 13: SALES SECTION GENERATION
# =============================================================================
# Now uses add_formatted_table for BOTH weekly and monthly tables
# Now includes source file display below tables

def create_sales_section(doc, source_log):
    """Create Sales Metrics section."""
    
    # Section 2 header
    h = doc.add_heading("Section 2: Sales Metrics", level=1)
    insert_page_break_before(h)
    
    # Weekly sales metrics table
    weekly_path = os.path.join(OUTPUTS_PATH, 'sales_metrics_analysis/weekly_sales_metrics.xlsx')
    weekly_df = load_excel_metrics(weekly_path)
    if weekly_df is not None:
        source_log.append(f"\n--- Sales Weekly Table ---")
        add_formatted_table(
            doc, weekly_df, 
            title="Core Metrics [Pro Sales] — Weekly Overview (WoW)", 
            color_wow=True,
            source_file="outputs/sales_metrics_analysis/weekly_sales_metrics.xlsx"
        )
    
    # Monthly table (with page break) - NOW USES add_formatted_table for color coding
    monthly_path = os.path.join(OUTPUTS_PATH, 'sales_metrics_analysis/monthly_sales_metrics.xlsx')
    monthly_df = load_excel_metrics(monthly_path)
    if monthly_df is not None:
        source_log.append(f"\n--- Sales Monthly Table ---")
        # Create title paragraph with page break
        p = doc.add_paragraph()
        run = p.add_run("Core Metrics [Pro Sales] — Monthly Overview (MoM)")
        run.bold = True
        run.font.size = Pt(14)
        p.paragraph_format.space_after = Pt(6)
        insert_page_break_before(p)
        
        # Add table with color formatting (no title since we added it above)
        add_formatted_table(
            doc, monthly_df, 
            title=None, 
            color_wow=True,
            source_file="outputs/sales_metrics_analysis/monthly_sales_metrics.xlsx"
        )
    
    # LEAD TIME & DEAL UPGRADE
    h = doc.add_heading("Lead Time & Deal Upgrade", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Lead Time & Deal Upgrade ---")
    add_chart_with_insights(doc, 'Average Lead Time', CHART_CONFIG['Average Lead Time'], source_log)
    add_chart_with_insights(doc, 'Deal Upgrade Visits', CHART_CONFIG['Deal Upgrade Visits'], source_log)
    add_chart_with_insights(doc, 'Deal Upgrade CVR', CHART_CONFIG['Deal Upgrade CVR'], source_log, page_break_before=True)
    
    # PRO SITE FUNNEL
    h = doc.add_heading("Pro Site Funnel", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Pro Site ---")
    add_chart_with_insights(doc, 'Pro Site Visits', CHART_CONFIG['Pro Site Visits'], source_log)
    add_chart_with_insights(doc, 'Pro Site CVR', CHART_CONFIG['Pro Site CVR'], source_log)
    
    # BOOKED CALLS
    h = doc.add_heading("Booked Calls", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Booked Calls ---")
    add_chart_with_insights(doc, 'Booked Calls - Closers (Discovery Call)', CHART_CONFIG['Booked Calls - Closers (Discovery Call)'], source_log)
    add_chart_with_insights(doc, 'Booked Calls - Setters (Intro Call)', CHART_CONFIG['Booked Calls - Setters (Intro Call)'], source_log)
    
    # SALES PIPELINE
    h = doc.add_heading("Sales Pipeline: Calls", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Sales Pipeline Calls ---")
    add_chart_with_insights(doc, 'Scheduled Calls', CHART_CONFIG['Scheduled Calls'], source_log)
    add_chart_with_insights(doc, 'Live Calls', CHART_CONFIG['Live Calls'], source_log)
    add_chart_with_insights(doc, 'Show Rate', CHART_CONFIG['Show Rate'], source_log, page_break_before=True)
    
    # OFFERS
    h = doc.add_heading("Offers", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Offers ---")
    add_chart_with_insights(doc, 'Offers', CHART_CONFIG['Offers'], source_log)
    add_chart_with_insights(doc, 'Offer Rate', CHART_CONFIG['Offer Rate'], source_log)
    
    # CLOSES
    h = doc.add_heading("Closes", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Closes ---")
    add_chart_with_insights(doc, 'Closes', CHART_CONFIG['Closes'], source_log)
    add_chart_with_insights(doc, 'Offer to Close Rate', CHART_CONFIG['Offer to Close Rate'], source_log)
    
    return doc


print("✓ Sales section function loaded (with table source info)")


✓ Sales section function loaded (with table source info)


### Section 14: Main Report Generation

In [29]:
# =============================================================================
# SECTION 14: MAIN REPORT GENERATION
# =============================================================================
# Orchestrates the complete report generation process
# Sets the reporting week filter before processing any data

def generate_wbr_report():
    """
    Generate the complete Weekly Business Review report.
    
    Process:
    1. Load reference dates from analysis_ref_date.csv
    2. Set reporting week end date for data filtering
    3. Load weekly metrics tables for executive summary
    4. Create document with cover page
    5. Add executive summary
    6. Add newsletter section (tables + charts + insights)
    7. Add sales section (tables + charts + insights)
    8. Add page numbers
    9. Save DOCX to reports folder
    """
    print("=" * 60)
    print("GENERATING WEEKLY BUSINESS REVIEW (WBR) REPORT")
    print("=" * 60)
    
    source_log = []
    source_log.append("\n" + "=" * 60)
    source_log.append("DATA SOURCES USED IN THIS REPORT")
    source_log.append("=" * 60)
    
    # Step 1: Load reference dates
    print("\n[Step 1] Loading reference dates...")
    date_info = parse_reference_dates(REF_DATE_FILE)
    print(f"  Reporting week: {date_info['date_range_display']}")
    source_log.append(f"\nReference Date File: analysis_ref_date.csv")
    source_log.append(f"  Reporting week: {date_info['date_range_display']}")
    
    # Step 2: Set reporting week end date for filtering
    print("\n[Step 2] Setting data filter...")
    set_reporting_week_end(date_info)
    
    # Step 3: Load metrics tables
    print("\n[Step 3] Loading metrics tables...")
    newsletter_weekly_path = os.path.join(OUTPUTS_PATH, 'newsletter_analysis/weekly_newsletter_metrics.xlsx')
    sales_weekly_path = os.path.join(OUTPUTS_PATH, 'sales_metrics_analysis/weekly_sales_metrics.xlsx')
    
    newsletter_df = load_excel_metrics(newsletter_weekly_path)
    sales_df = load_excel_metrics(sales_weekly_path)
    
    print(f"  Newsletter metrics: {'✓ Loaded' if newsletter_df is not None else '⚠ Not found'}")
    print(f"  Sales metrics: {'✓ Loaded' if sales_df is not None else '⚠ Not found'}")
    
    # Step 4: Create document
    print("\n[Step 4] Creating document...")
    doc = Document()
    
    # Step 5: Cover page
    print("\n[Step 5] Adding cover page...")
    doc = create_cover_page(doc, date_info)
    
    # Step 6: Executive Summary
    print("\n[Step 6] Adding executive summary...")
    doc = create_executive_summary(doc, newsletter_df, sales_df)
    
    # Step 7: Newsletter section
    print("\n[Step 7] Adding newsletter section...")
    doc = create_newsletter_section(doc, source_log)
    
    # Step 8: Sales section
    print("\n[Step 8] Adding sales section...")
    doc = create_sales_section(doc, source_log)
    
    # Step 9: Add page numbers
    print("\n[Step 9] Adding page numbers...")
    add_page_numbers(doc)
    
    # Step 10: Save document
    print("\n[Step 10] Saving document...")
    filename = f"Weekly_Business_Review_{date_info['date_range_filename']}.docx"
    output_path = os.path.join(REPORTS_PATH, filename)
    doc.save(output_path)
    
    print(f"\n{'=' * 60}")
    print(f"✓ REPORT GENERATED SUCCESSFULLY!")
    print(f"{'=' * 60}")
    print(f"  File: {filename}")
    print(f"  Location: {output_path}")
    
    print("\n" + "\n".join(source_log))
    
    return output_path


print("✓ Main report generation function loaded")

✓ Main report generation function loaded


### Section 15: Execute Report Generation

In [30]:
# =============================================================================
# SECTION 15: EXECUTE REPORT GENERATION
# =============================================================================

# Generate the report
report_path = generate_wbr_report()

print(f"\n\n{'=' * 60}")
print("REPORT COMPLETE!")
print("=" * 60)
print(f"Open: {report_path}")

GENERATING WEEKLY BUSINESS REVIEW (WBR) REPORT

[Step 1] Loading reference dates...
  Reporting week: 11/30/2025 to 12/06/2025

[Step 2] Setting data filter...
  Data filter: Only including data up to 12/06/2025

[Step 3] Loading metrics tables...
  Newsletter metrics: ✓ Loaded
  Sales metrics: ✓ Loaded

[Step 4] Creating document...

[Step 5] Adding cover page...

[Step 6] Adding executive summary...

[Step 7] Adding newsletter section...

[Step 8] Adding sales section...

[Step 9] Adding page numbers...

[Step 10] Saving document...

✓ REPORT GENERATED SUCCESSFULLY!
  File: Weekly_Business_Review_11-30-2025_to_12-06-2025.docx
  Location: /Users/mariaangelicabaldres/Desktop/smb_dh_metrics/reports/Weekly_Business_Review_11-30-2025_to_12-06-2025.docx


DATA SOURCES USED IN THIS REPORT

Reference Date File: analysis_ref_date.csv
  Reporting week: 11/30/2025 to 12/06/2025

--- Newsletter Weekly Table ---

--- Newsletter Monthly Table ---

--- Landing Page Funnel Charts ---
  Chart: newsle

#